# Final OCSD (harmonised M4): one scene step by step + same sketch with other backgrounds (for the slides)

Same scene as the module slides (2 orange buses + 1 wooden bench near a train station), seed 0, with the **final OCSD**: SD 1.5, all five modules M1-M5, M3 on, harmonised M4 frozen in `results/m4_study_v1/locked.json` (alpha 0.2, release 0.4, scene ControlNet 0.6, LoRA from 45%).
Saves the U-Net input and the predicted clean image at steps 1, 6, 12, 18, 24 and 30, the M4 anchor weight of every step, the M2 candidate scores, and the image with the own background plus 3 other backgrounds.

**How to run:** Runtime -> Change runtime type -> any GPU (T4, L4 or A100) -> **Run all**. About 15 minutes. A ZIP downloads at the end; it is also on Drive at `KLTN_OCSD/trace_ocsd/bgswap_qd_3_medium_003_s0_final.zip`.

The last cell also packs the paper images of the 36 review scenes (seed 0) for the result slides: `slide_images_m4_study_v1.zip`.


In [ ]:
#@title 1. Settings
SID    = "qd_3_medium_003"   #@param {type:"string"}
SEED   = 0                   #@param {type:"integer"}
BGS    = ["on a city street", "on a snowy field", "on a sandy beach"]
BRANCH = "claude/project-thread-srtlwl"  #@param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/KLTN_OCSD"  #@param {type:"string"}


In [ ]:
#@title 2. Drive, GPU check, code, libraries
import os, subprocess, sys, shutil
from google.colab import drive
drive.mount("/content/drive")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], capture_output=True, text=True)
print(gpu.stdout)
if gpu.returncode or int(gpu.stdout.strip().split(",")[-1]) < 14000:
    raise RuntimeError("This needs a GPU with at least 14 GB (T4, L4 or A100): Runtime > Change runtime type > GPU, then Run all")
# OCSD on SD 1.5 (with M3 training) fits in about 12 GB, so a T4, L4 or A100 all work
CODE = "/content/KLTN_OCSD_final_trace"
shutil.rmtree(CODE, ignore_errors=True)
subprocess.run(["git", "clone", "-q", "-b", BRANCH, "https://github.com/bobbibao/ocsd-thesis.git", CODE], check=True)
print("Code:", subprocess.run(["git", "-C", CODE, "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)
sys.path.insert(0, CODE)
for _m in [m for m in sys.modules if m == "ocsd" or m.startswith("ocsd.")]: del sys.modules[_m]
os.environ["HF_HOME"] = "/content/hf_cache"
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(CODE, "requirements-colab.txt")], capture_output=True, text=True)
print(r.stdout[-1000:], r.stderr[-2000:])
if r.returncode:
    raise RuntimeError("pip install failed (see the text above)")
from ocsd import hfcache
hfcache.repair(os.environ["HF_HOME"])

In [ ]:
#@title 3. Final OCSD: steps + own background + 3 others (about 10 minutes after the models are downloaded)
from ocsd import bg_swap
FINAL = bg_swap.final_overrides(CODE)
print("final OCSD settings:", FINAL)
META = bg_swap.run(DRIVE_ROOT, sid=SID, seed=SEED, bgs=BGS, overrides=FINAL, tag="_final", record_steps=(0, 5, 11, 17, 23, 29), attn_steps=(11, 23))
for r in META["runs"]: print(r["bg"], "|", r["global_prompt"], "|", r["seconds"], "s, tries", r["log"].get("tries"))
print("M4 anchor weights:", META["steps"]["m4"])


In [ ]:
#@title 4. Show the result and download the ZIP
from IPython.display import Image, display
out = os.path.join(DRIVE_ROOT, "trace_ocsd", f"bgswap_{SID}_s{SEED}_final")
display(Image(os.path.join(out, "SHEET.jpg"), width=1400))
from google.colab import files
try:
    files.download(META["zip"])
except Exception as exc:
    print("Get it from Drive:", META["zip"], exc)
print("Done. Disconnect the GPU runtime.")


In [ ]:
#@title 5. Paper images for the result slides (36 review scenes, seed 0, 5 methods + sketch), small JPGs in one ZIP
import json as _json, zipfile, glob
from PIL import Image as _Im
P = _json.load(open(os.path.join(CODE, "assets/m4_study_v1/protocol.json")))
OUT = os.path.join(DRIVE_ROOT, "m4_study_v1", "paper", "outputs", "quickdraw")
BENCH = os.path.join(DRIVE_ROOT, "data", "benchmarks", "quickdraw")
zpath = os.path.join(DRIVE_ROOT, "trace_ocsd", "slide_images_m4_study_v1.zip")
n = 0
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for sid in P["paper"]["review"]["scenes"]:
        for arm in ["ocsd_final", "ocsd", "gligen", "controlnet", "collage"]:
            src = os.path.join(OUT, arm, f"{sid}_s0.png")
            if os.path.exists(src):
                tmp = "/content/_s.jpg"; _Im.open(src).convert("RGB").resize((384, 384)).save(tmp, quality=90)
                z.write(tmp, f"{arm}/{sid}_s0.jpg"); n += 1
            else:
                print("missing", src)
        for name in ["sketch.png", "scene.json"]:
            src = os.path.join(BENCH, sid, name)
            if os.path.exists(src):
                z.write(src, f"scenes/{sid}/{name}")
print(n, "images ->", zpath)
try:
    files.download(zpath)
except Exception as exc:
    print("Get it from Drive:", zpath, exc)
